# Homework 5: Monitoring a RAG system with OpenTelemetry

This notebook follows the [2026 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md).
It reuses the course's `RAGBase` and starter design, instruments the three RAG
operations, saves spans to SQLite, and derives every multiple-choice answer from
the collected data.

The notebook expects `OPENAI_API_KEY` in the repository's `.env` file. It never
prints or stores the key. Install the project environment with `uv sync` before
running if needed.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from openai import OpenAI
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)

ENV_FILE_STR = find_dotenv(usecwd=True)
if not ENV_FILE_STR:
    raise FileNotFoundError("Create a .env file containing OPENAI_API_KEY")
ENV_FILE = Path(ENV_FILE_STR)
load_dotenv(ENV_FILE)
PROJECT_ROOT = ENV_FILE.parent

QUERY = "How does the agentic loop keep calling the model until it stops?"
MODEL = "gpt-5.4-mini"
DB_PATH = PROJECT_ROOT / "hw/hv05/traces.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
DB_PATH.unlink(missing_ok=True)  # make repeated full-notebook runs reproducible

# Standard API prices for gpt-5.4-mini at the time of the homework, per 1M tokens.
INPUT_PRICE = 0.75
OUTPUT_PRICE = 4.50

## 1. Load and index the course lessons

This is the starter's data-loading code. The commit is pinned by the homework,
so all runs search the same 72 lesson pages.

In [2]:
COMMIT = "8c1834d"
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)
print(f"Indexed {len(documents)} lesson pages")

Indexed 72 lesson pages


## 2. Reuse the course RAG implementation

`RAGBase` is reproduced from the official `rag_helper.py` so the notebook is
self-contained. `llm()` deliberately returns the raw Responses API object; the
traced subclass needs its usage fields.

In [3]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class RAGBase:
    def __init__(self, index, llm_client, instructions=INSTRUCTIONS,
                 prompt_template=PROMPT_TEMPLATE, model=MODEL):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.extend([doc["filename"], doc["content"], ""])
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        return self.prompt_template.format(
            question=query, context=self.build_context(search_results)
        )

    def llm(self, prompt):
        messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        return self.llm_client.responses.create(model=self.model, input=messages)

    def rag(self, query):
        results = self.search(query)
        response = self.llm(self.build_prompt(query, results))
        return response.output_text

## 3. SQLite exporter and OpenTelemetry setup

Each completed span is inserted into `spans`. OpenTelemetry timestamps are
nanoseconds since the Unix epoch, so `(end_time - start_time) / 1e6` is the
duration in milliseconds.

In [4]:
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (span.name, span.start_time, span.end_time,
                 attrs.get("input_tokens"), attrs.get("output_tokens"),
                 attrs.get("cost")),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        self.conn.commit()
        return True


provider = TracerProvider()
exporter = SQLiteSpanExporter(DB_PATH)
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

## 4. Instrument `rag`, `search`, and `llm`

Because `rag()` calls the other two methods while its span is current, `search`
and `llm` automatically become child spans. Token counts and estimated cost are
attached only to `llm`, where the response usage is available.

In [5]:
class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            cost = (
                usage.input_tokens * INPUT_PRICE
                + usage.output_tokens * OUTPUT_PRICE
            ) / 1_000_000
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", cost)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)


rag = RAGTraced(index=index, llm_client=OpenAI(), model=MODEL)

## Q1-Q4: Run the first trace and inspect it

In [6]:
answer = rag.rag(QUERY)
print(answer)

with sqlite3.connect(DB_PATH) as conn:
    first_trace = pd.read_sql_query(
        """SELECT name, input_tokens, output_tokens, cost,
                  (end_time - start_time) / 1e6 AS duration_ms
           FROM spans ORDER BY start_time""",
        conn,
    )
first_trace

The loop keeps calling the model with a `while True` loop and stops when the model returns no `function_call` items.

In the code:

- set `has_function_calls = False`
- call the model
- if the response includes any `function_call`, run the tool and set `has_function_calls = True`
- after processing the response, `break` only if `has_function_calls == False`

So the model keeps getting called again and again until it gives a final answer with no more tool calls.


,name,input_tokens,output_tokens,cost,duration_ms
0,rag,NaN,NaN,NaN,5366.144321
1,search,NaN,NaN,NaN,5.868305
2,llm,7111.0,109.0,0.005824,5334.032792


In [7]:
q1_span_count = len(first_trace)
q2_input_tokens = int(first_trace.loc[first_trace.name == "llm", "input_tokens"].iloc[0])
q3_llm_ms = float(first_trace.loc[first_trace.name == "llm", "duration_ms"].iloc[0])
q4_span_names = sorted(first_trace.name.unique())

print(f"Q1: {q1_span_count} spans")
print(f"Q2: {q2_input_tokens:,} input tokens -> closest option: 7,000")
print(f"Q3: LLM duration = {q3_llm_ms:,.0f} ms")
print(f"Q4: span names = {q4_span_names}")

Q1: 3 spans
Q2: 7,111 input tokens -> closest option: 7,000
Q3: LLM duration = 5,334 ms
Q4: span names = ['llm', 'rag', 'search']


### Answers Q1-Q4

1. **3 spans.** One `rag` parent plus its `search` and `llm` children.
2. **7,000 input tokens (closest option).** The exact measured count is printed
   above and can vary slightly if the upstream content or provider changes.
3. **Over 2000 ms.** The exact duration is measured above; latency varies by run.
4. **`rag`, `search`, and `llm`.** These are precisely the three methods wrapped
   by `RAGTraced`.

## Q5: Aggregate child-span durations

Run the same query once more. Excluding `rag` avoids double-counting because the
parent duration already contains both child operations.

In [8]:
_ = rag.rag(QUERY)

with sqlite3.connect(DB_PATH) as conn:
    duration_totals = pd.read_sql_query(
        """SELECT name,
                  COUNT(*) AS calls,
                  SUM(end_time - start_time) / 1e6 AS total_duration_ms
           FROM spans
           WHERE name <> 'rag'
           GROUP BY name
           ORDER BY total_duration_ms DESC""",
        conn,
    )
duration_totals

,name,calls,total_duration_ms
0,llm,2,7346.626372
1,search,2,8.095980


**Q5 answer: `llm`.** Network model inference takes far more total time than the
in-memory minsearch lookup, as the aggregation above demonstrates.

## Q6: Token stability over four identical runs

Two calls already exist (the Q1-Q4 call and the extra Q5 call). Run two more to
reach four total, then compare only the `llm` rows.

In [9]:
for _ in range(2):
    rag.rag(QUERY)

with sqlite3.connect(DB_PATH) as conn:
    token_runs = pd.read_sql_query(
        """SELECT rowid AS run, input_tokens
           FROM spans
           WHERE name = 'llm'
           ORDER BY start_time""",
        conn,
    )

minimum = int(token_runs.input_tokens.min())
maximum = int(token_runs.input_tokens.max())
variation_pct = (maximum - minimum) / minimum * 100
display(token_runs)
print(f"Range: {minimum:,} to {maximum:,} tokens; variation = {variation_pct:.2f}%")

,run,input_tokens
0,2,7111
1,5,7111
2,8,7111
3,11,7111


Range: 7,111 to 7,111 tokens; variation = 0.00%


**Q6 answer: They are identical.** Deterministic text search retrieves the same
five documents for an unchanged query, so the constructed prompt—and therefore
its input-token count—is unchanged across all four calls.

## Final answers

| Question | Answer |
|---|---|
| Q1 | **3** |
| Q2 | **7,000** |
| Q3 | **Over 2000 ms** |
| Q4 | **`rag`, `search`, and `llm`** |
| Q5 | **`llm`** |
| Q6 | **They are identical** |

Timing and exact token totals are empirical. Re-running the notebook preserves
the answer categories while refreshing the displayed measurements.